# Class Work 1: Advanced Dimensionality Reduction

Welcome to the first tutorial. In this notebook, we will explore the curse of dimensionality, advanced distance metrics, and linear/statistical dimensionality reduction techniques such as SVD, ICA, Kernel PCA, and LDA. 

This notebook contains code completions. Look for `___` or `TODO` to complete the exercises. This should take approximately 1 hour.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, TruncatedSVD, FastICA, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.datasets import load_iris, make_circles
from scipy.spatial.distance import cdist


## 1. The Curse of Dimensionality
In high dimensions, the concept of distance collapses. The distance between the closest points and the farthest points becomes almost identical. Let's demonstrate this by calculating distances between random points in increasing dimensions.

In [ ]:
dimensions = [2, 10, 50, 100, 500, 1000, 5000]
ratios = []

for dim in dimensions:
    # Generate 100 random points in 'dim' dimensions
    points = np.random.rand(100, dim)

    # Calculate pairwise Euclidean distances using cdist
    distances = cdist(points, points, metric='euclidean')

    # Get upper triangle to ignore self-distances (0)
    dist_upper = distances[np.triu_indices(100, k=1)]

    d_max = np.max(dist_upper)
    d_min = np.min(dist_upper)

    # Calculate the ratio (d_max - d_min) / d_min
    ratio = (d_max - d_min) / d_min
    ratios.append(ratio)

plt.plot(dimensions, ratios, marker='o')
plt.title('Curse of Dimensionality: Distance Collapse')
plt.xlabel('Dimension')
plt.ylabel('(D_max - D_min) / D_min')
plt.show()

## 2. SVD vs PCA
Instead of calculating the covariance matrix (which can be numerically unstable and memory-heavy), modern implementations use Singular Value Decomposition (SVD) directly on the centered data matrix. Let's extract the principal components using TruncatedSVD.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

# Center the data by subtracting the mean of each feature
X_centered = X - np.mean(X, axis=0)

# Initialize TruncatedSVD to extract 2 components
svd = TruncatedSVD(n_components=2)

# Fit and transform the centered data
X_svd = svd.fit_transform(X_centered)

plt.scatter(X_svd[:, 0], X_svd[:, 1], c=y, cmap='viridis')
plt.title('SVD Projection of Iris Dataset')
plt.xlabel('Component 1 (Scores)')
plt.ylabel('Component 2 (Scores)')
plt.show()

## 3. Independent Component Analysis (ICA)
PCA looks for orthogonal directions of maximum variance. ICA looks for statistically independent sources by maximizing non-Gaussianity (negentropy). Let's unmix two signals.

In [ ]:
# Generate mixed signals
time = np.linspace(0, 8, 2000)
s1 = np.sin(2 * time)  # Signal 1 : sinusoidal signal
s2 = np.sign(np.sin(3 * time))  # Signal 2 : square signal
S = np.c_[s1, s2]

# Mix data
A = np.array([[1, 1], [0.5, 2]])  # Mixing matrix
X_mixed = np.dot(S, A.T)  # Generate observations

# Initialize FastICA with 2 components
ica = FastICA(n_components=2, random_state=42)

# Fit and transform the mixed data to recover sources
S_recovered = ica.fit_transform(X_mixed)

fig, models = plt.subplots(3, 1, figsize=(8, 6))
models[0].plot(S); models[0].set_title('Original Independent Sources')
models[1].plot(X_mixed); models[1].set_title('Mixed Signals (What we observe)')
models[2].plot(S_recovered); models[2].set_title('ICA Recovered Signals')
plt.tight_layout()
plt.show()

## 4. Kernel PCA
When data is not linearly separable, we can use the kernel trick to project it into a higher dimension where it becomes linearly separable. The RBF kernel is a popular choice.

In [ ]:
X_circ, y_circ = make_circles(n_samples=400, factor=.3, noise=.05, random_state=42)

# Apply standard PCA (2 components) and see how it fails
pca_circ = PCA(n_components=2)
X_pca_circ = pca_circ.fit_transform(X_circ)

# Apply Kernel PCA with 'rbf' kernel and gamma=10
kpca = KernelPCA(kernel='rbf', gamma=10, n_components=2)
X_kpca = kpca.fit_transform(X_circ)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='viridis'); ax1.set_title('Original Space')
ax2.scatter(X_pca_circ[:, 0], X_pca_circ[:, 1], c=y_circ, cmap='viridis'); ax2.set_title('Standard PCA')
ax3.scatter(X_kpca[:, 0], X_kpca[:, 1], c=y_circ, cmap='viridis'); ax3.set_title('Kernel PCA (RBF)')
plt.show()

## 5. Linear Discriminant Analysis (LDA)
LDA maximizes class separation by maximizing the ratio of between-class variance to within-class variance. Note that it is limited to (number of classes - 1) dimensions.

In [ ]:
# We will use the Iris dataset from before (X, y)

# Initialize LDA
lda = LinearDiscriminantAnalysis(n_components=2)

# Fit and transform data. Remember LDA is supervised, so it needs 'y'!
X_lda = lda.fit_transform(X, y)

plt.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='viridis')
plt.title('LDA Projection of Iris Dataset')
plt.xlabel('LD 1')
plt.ylabel('LD 2')
plt.show()